# PRACTICA OBLIGATORIA TEAM CHALLENGE SQL MURDER MISTERY 

In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect("../sql-murder-mystery.db")

In [3]:
query = """
SELECT *
FROM crime_scene_report
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)
df

,date,type,description,city
0,20180115,robbery,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,murder,Life? Dont talk to me about life.,Albany
2,20180115,murder,"Mama, I killed a man, put a gun against his he...",Reno
3,20180215,murder,REDACTED REDACTED REDACTED,SQL City
4,20180215,murder,Someone killed the guard! He took an arrow to ...,SQL City
5,20180115,theft,Big Bully stole my lunch money!,Chicago
6,20180115,fraud,"Lorem ipsum dolor sit amet, consectetur adipis...",Seattle
7,20170712,theft,"A lone hunter stalks the night, firing arrows ...",SQL City
8,20170820,arson,"Wield the Hammer of Sol with honor, Titan, it ...",SQL City
9,20171110,robbery,The Gjallarhorn shoulder-mounted rocket system...,SQL City


# 🕵️ Investigación — Paso 1

## Pista:

El asesinato ocurrió el 15 de enero de 2018 en SQL City.

## Objetivo:

Recuperar el informe de la escena del crimen.

## Query:

In [4]:
query = """
SELECT *
FROM crime_scene_report
WHERE date = 20180115
  AND type = 'murder'
  AND city = 'SQL City';
"""

crime_scene = pd.read_sql_query(query, conn)
crime_scene

,date,type,description,city
0,20180115,murder,Security footage shows that there were 2 witne...,SQL City


## INFORME — Recuperar el informe de la escena del crimen

El detective indica que el asesinato ocurrió el 15 de enero de 2018
en SQL City.

Primero identificamos la tabla `crime_scene_report`, que contiene la
fecha, el tipo de crimen, la descripción y la ciudad.

La fecha se almacena como un entero en formato `YYYYMMDD`, por lo que
el 15 de enero de 2018 corresponde a `20180115`.

Aplicamos las tres pistas proporcionadas:

- Fecha: `20180115`
- Tipo: `murder`
- Ciudad: `SQL City`

Después de ejecutar la query, el resultado de esa consulta es la siguiente pista de nuestra investigación.

### Resultado

La consulta devuelve un único informe de asesinato ocurrido el
15 de enero de 2018 en SQL City.

Para continuar la investigación necesitamos analizar la descripción
completa del informe, ya que contiene las pistas proporcionadas por
la escena del crimen.

In [5]:
print(crime_scene.iloc[0]["description"])

Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".


### El informe dice:

Hay 2 testigos.

El primero vive en la última casa de Northwestern Dr.
La segunda se llama Annabel y vive en algún lugar de Franklin Ave.

Esto nos da dos investigaciones independientes.

# 🕵️ Paso 2 — Encontrar a los dos testigos

### Testigo 1

La pista dice:

~~~
"The first witness lives at the last house on Northwestern Dr."
~~~

¿Qué significa "last house"?

Tenemos que buscar la persona que tenga el número de dirección más alto de esa calle.

No sabemos todavía cuál es ese número, así que no debemos inventarlo.

La query correcta es:

~~~
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
LIMIT 1;
~~~

### ¿Qué estamos haciendo?

~~~ 
WHERE address_street_name = 'Northwestern Dr'
~~~

Nos quedamos únicamente con las personas que viven en esa calle.

~~~
ORDER BY address_number DESC
~~~

Ordenamos las casas de mayor a menor número.

~~~
LIMIT 1
~~~

Nos quedamos con la primera, que será la casa con el número más alto.

In [6]:
query = """
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
LIMIT 1;
"""

witness_1 = pd.read_sql_query(query, conn)
witness_1

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


### Testigo 2

La segunda pista es:

~~~
"The second witness, named Annabel, lives somewhere on Franklin Ave."
~~~

Aquí tenemos dos condiciones:

~~~
name = Annabel
address_street_name = Franklin Ave
~~~

Por tanto:

~~~
SELECT *
FROM person
WHERE name = 'Annabel'
  AND address_street_name = 'Franklin Ave';
  ~~~

In [7]:
query = """
SELECT *
FROM person
WHERE name = 'Annabel'
  AND address_street_name = 'Franklin Ave';
"""

witness_2 = pd.read_sql_query(query, conn)
witness_2

,id,name,license_id,address_number,address_street_name,ssn


## Resultados de esta segunda investigación: 

### Paso 2 — Identificación de los testigos

El informe de la escena del crimen indica que existen dos testigos.

Para localizar al primer testigo buscamos la persona que vive en la
última casa de `Northwestern Dr`. Para ello filtramos por la calle,
ordenamos `address_number` de forma descendente y seleccionamos la
primera fila.

El resultado identifica a:

- **Morty Schapiro**
- ID: `14887`
- Dirección: `4919 Northwestern Dr`

Para el segundo testigo, el informe indica que se llama Annabel y que
vive en `Franklin Ave`. La consulta combinando ambos criterios no
devolvió resultados, por lo que realizamos una búsqueda adicional por
nombre para comprobar si Annabel existe en la tabla `person`.



## Paso 2.1 — Buscar a Annabel

In [8]:
query = """
SELECT *
FROM person
WHERE name = 'Annabel';
"""

annabel = pd.read_sql_query(query, conn)
annabel

,id,name,license_id,address_number,address_street_name,ssn


### Segunda comprobación 

Vamos a comprobar si existe alguien en Franklin Ave

In [9]:
query = """
SELECT *
FROM person
WHERE address_street_name = 'Franklin Ave';
"""

franklin_residents = pd.read_sql_query(query, conn)
franklin_residents

,id,name,license_id,address_number,address_street_name,ssn
0,12207,Wilmer Wolever,509484,139,Franklin Ave,636825374
1,16371,Annabel Miller,490173,103,Franklin Ave,318771143
2,17683,Johnnie Schee,968887,1277,Franklin Ave,815977821
3,18651,Carleen Etoll,356746,22,Franklin Ave,193369255
4,22636,Zachary Ybarbo,768359,785,Franklin Ave,285346605
5,24737,Gema Nantz,273410,3968,Franklin Ave,180545802
6,30654,Clarita Rickels,418084,2254,Franklin Ave,714941023
7,32264,Shelby Dezeeuw,735415,1391,Franklin Ave,143197463
8,33793,Amado Mattan,161915,99,Franklin Ave,125205748
9,34592,Cordell Lindamood,592762,3657,Franklin Ave,509890333


### Perfecto. Ahora vemos claramente qué ha ocurrido: Annabel sí existe, pero el problema está en que su nombre completo es `Annabel Miller` , no simplemente `Annabel`.

Por tanto, la pista del informe:

```
The second witness, named Annabel, lives somewhere on "Franklin Ave".
```

corresponde a `Annabel Miller`.

### Hay una pequeña lección SQL interesante aquí.

Nuestra primera consulta fue:

``` 
WHERE name = 'Annabel'
``` 

y no encontró nada.

¿Por qué?

Porque `name` parece almacenar el nombre completo, no únicamente el nombre de pila.

Lo podemos comprobar con la fila:

`Annabel Miller`

Por tanto, una consulta más adecuada habría sido:

```
SELECT *
FROM person
WHERE name LIKE 'Annabel%'
  AND address_street_name = 'Franklin Ave';
  ```
Esta consulta busca nombres que empiecen por Annabel.

### Entonces la consulta correcta deberia haber sido la siguiente: 

In [10]:
query = """
SELECT *
FROM person
WHERE name LIKE 'Annabel%'
  AND address_street_name = 'Franklin Ave';
"""

annabel = pd.read_sql_query(query, conn)
annabel

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


# 🕵️ Paso 3 — Interrogar a los testigos

## Podríamos hacer dos queries separadas:

### Entrevista de `Morty`

In [11]:
query = """
SELECT *
FROM interview
WHERE person_id = 14887;
"""

morty_interview = pd.read_sql_query(query, conn)
morty_interview

,person_id,transcript
0,14887,I heard a gunshot and then saw a man run out. ...


### Entrevista de `Annabel`

In [12]:
query = """
SELECT *
FROM interview
WHERE person_id = 16371;
"""

annabel_interview = pd.read_sql_query(query, conn)
annabel_interview

,person_id,transcript
0,16371,"I saw the murder happen, and I recognized the ..."


### Pero aquí podemos hacer algo ligeramente mejor: consultar las dos entrevistas en una sola query, porque ya conocemos los IDs de ambos testigos.

In [13]:
query = """
SELECT
    p.id,
    p.name,
    i.transcript
FROM person AS p
JOIN interview AS i
    ON p.id = i.person_id
WHERE p.id IN (14887, 16371);
"""

witness_interviews = pd.read_sql_query(query, conn)
witness_interviews

,id,name,transcript
0,14887,Morty Schapiro,I heard a gunshot and then saw a man run out. ...
1,16371,Annabel Miller,"I saw the murder happen, and I recognized the ..."


## Resultados de la tercera investigación

### Paso 3 — Entrevistas de los testigos

Una vez identificados los dos testigos, utilizamos sus identificadores
en la tabla `interview`.

La tabla `interview` relaciona cada declaración con una persona
mediante `person_id`.

Realizamos un `JOIN` entre `person` e `interview` para obtener el nombre
del testigo junto con su declaración.

### Debido a que pandas sigue truncando las salidas con ... vamos a realizar la siguiente consulta para obtener la información completa de las entrevistas: 


In [14]:
query = """
SELECT
    p.name,
    i.transcript
FROM person AS p
JOIN interview AS i
    ON p.id = i.person_id
WHERE p.id IN (14887, 16371)
ORDER BY p.id;
"""

witness_interviews = pd.read_sql_query(query, conn)

for _, row in witness_interviews.iterrows():
    print(f"\n=== ENTREVISTA DE {row['name'].upper()} ===")
    print(row["transcript"])


=== ENTREVISTA DE MORTY SCHAPIRO ===
I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".

=== ENTREVISTA DE ANNABEL MILLER ===
I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.


# 🕵️ Paso 4 — Analizar las declaraciones

### `Morty Schapiro`

Su declaración nos da tres pistas:

El asesino era un `hombre`.
Llevaba una bolsa de `Get Fit Now Gym`.
El número de socio empezaba por `48Z`.
Era `Gold member`.
Escapó en un coche cuya matrícula incluía `H42W`.

La parte importante para nosotros es: 
`
membership number → empieza por 48Z
membership status → gold
plate number → contiene H42W
`

`Annabel Miller`

Annabel nos proporciona una pista completamente diferente:

Reconoció al asesino de su gimnasio cuando estaba entrenando el `9 de enero`.

### Por tanto, ahora tenemos dos líneas de investigación:

                         ASESINATO
                             │
              ┌──────────────┴──────────────┐
              │                             │
           Morty                         Annabel
              │                             │
       Get Fit Now Gym              Lo reconoció
              │                     del gimnasio
       48Z + Gold                    9 enero
              │
        matrícula H42W



# 🔎 Paso 5 — Investigar Get Fit Now Gym

El modelo nos dice que tenemos dos tablas:

``` 
get_fit_now_member

id
person_id
name
membership_start_date
membership_status


get_fit_now_check_in

membership_id
check_in_date
check_in_time
check_out_time
```

Esto nos viene perfecto.

Morty nos ha dicho:

El número de socio comenzaba por `48Z` y era `Gold`.

Por tanto, nuestro siguiente objetivo es encontrar qué socios cumplen esas condiciones.

## Primera consulta: socios cuyo ID empieza por 48Z y son Gold
### Vamos a consultar get_fit_now_member:

In [15]:
query = """
SELECT *
FROM get_fit_now_member
WHERE id LIKE '48Z%'
  AND membership_status = 'gold';
"""

gym_members = pd.read_sql_query(query, conn)

print(gym_members.to_string(index=False))

   id  person_id          name  membership_start_date membership_status
48Z7A      28819  Joe Germuska               20160305              gold
48Z55      67318 Jeremy Bowers               20160101              gold


## NUEVO INFORME

### Paso 5 — Análisis de las declaraciones

Las entrevistas proporcionan nuevas pistas sobre el asesino.

Morty Schapiro declara que vio salir corriendo a un hombre que llevaba
una bolsa del gimnasio "Get Fit Now Gym". El número de socio comenzaba
por `48Z` y el individuo era miembro Gold. También observó que el
vehículo utilizado por el sospechoso tenía una matrícula que incluía
`H42W`.

Annabel Miller declara que reconoció al asesino de su gimnasio cuando
estaba entrenando el 9 de enero.

A partir de las pistas de Morty, investigaremos primero los miembros de
"Get Fit Now Gym" cuyo identificador comienza por `48Z` y cuyo estado
de membresía es `gold`.

## El conjunto de sospechosos se ha reducido a dos personas:

```
48Z7A → Joe Germuska → person_id 28819
48Z55 → Jeremy Bowers → person_id 67318
```

# 🕵️ Paso 5.1  — Cruzar con la matrícula

Recordemos la otra pista que nos dio Morty:

The man got into a car with a plate that included "H42W".
Ahora tenemos que relacionar nuestros dos candidatos con la tabla `person` y después con `drivers_license`.

El modelo nos dice que:

`person.license_id → drivers_license.id`

y `drivers_license` contiene:

`plate_number`


### Por tanto, nuestra investigación queda:

```
Get Fit Now
     │
     ├── 48Z7A → Joe Germuska
     │
     └── 48Z55 → Jeremy Bowers
                    │
                    ▼
                  person
                    │
               license_id
                    │
                    ▼
            drivers_license
                    │
                    ▼
              plate_number

```

La pregunta que vamos a responder es:

¿Cuál de los dos candidatos tiene una matrícula que contiene H42W?

### 🔎 Nuestra siguiente query

In [16]:
query = """
SELECT
    p.id AS person_id,
    p.name,
    m.id AS membership_id,
    dl.plate_number
FROM get_fit_now_member AS m
JOIN person AS p
    ON m.person_id = p.id
JOIN drivers_license AS dl
    ON p.license_id = dl.id
WHERE m.id IN ('48Z7A', '48Z55')
  AND dl.plate_number LIKE '%H42W%';
"""

suspect = pd.read_sql_query(query, conn)

print(suspect.to_string(index=False))

 person_id          name membership_id plate_number
     67318 Jeremy Bowers         48Z55       0H42W2


### Los % están a ambos lados porque Morty no dijo que la matrícula empezara por H42W ni que terminara por H42W.

## Paso 5.1 — Comprobar la matrícula del vehículo

La declaración de Morty proporciona una segunda característica del
sospechoso: la matrícula del vehículo contenía `H42W`.

Hasta este momento tenemos dos candidatos que cumplen las condiciones
de la bolsa y membresía del gimnasio:

- Joe Germuska (`48Z7A`)
- Jeremy Bowers (`48Z55`)

Para comprobar la matrícula relacionamos `get_fit_now_member` con
`person` mediante `person_id` y posteriormente con `drivers_license`
mediante `license_id`.

Buscamos una matrícula que contenga `H42W` en cualquier posición.

La matrícula 0H42W2 contiene H42W, exactamente como declaró Morty.

Por tanto, podemos establecer:

Jeremy Bowers cumple todas las características aportadas por Morty hasta este momento.

### 🔎 Pero todavía NO hemos terminado

Aquí quiero que seamos rigurosos.

Tenemos una identificación muy fuerte:

Jeremy Bowers → candidato principal

Pero Annabel nos dio otra pista:

Ella reconoció al asesino de su gimnasio cuando estaba entrenando el 9 de enero.

Tenemos que comprobar esa declaración utilizando la base de datos.

# 🕵️ Paso 6 — Comprobar la coartada del gimnasio

`person_id` = `67318`
`membership_id` = `48Z55`

Ahora buscaremos sus entradas al gimnasio el:

9 de enero de 2018

In [17]:
query = """
SELECT
    membership_id,
    check_in_date,
    check_in_time,
    check_out_time
FROM get_fit_now_check_in
WHERE membership_id = '48Z55'
  AND check_in_date = 20180109;
"""

gym_checkin = pd.read_sql_query(query, conn)

print(gym_checkin.to_string(index=False))

membership_id  check_in_date  check_in_time  check_out_time
        48Z55       20180109           1530            1700


¿Qué estamos comprobando?

La pregunta concreta es:

¿Estuvo Jeremy Bowers en Get Fit Now Gym el 9 de enero de 2018?

Si aparece una entrada, la declaración de Annabel encaja con nuestra hipótesis.

Si no aparece, tendremos que reconsiderar la investigación.

## Paso 6 — Comprobar la presencia en el gimnasio

Annabel Miller declaró que reconoció al asesino de su gimnasio cuando
estaba entrenando el 9 de enero.

El candidato identificado hasta ahora es Jeremy Bowers, cuyo
`membership_id` es `48Z55`.

La tabla `get_fit_now_check_in` registra las entradas y salidas de los
miembros del gimnasio. Por ello, consultamos si el miembro `48Z55`
registró una entrada el 9 de enero de 2018 (`20180109`).

### Resultado

El miembro `48Z55`, correspondiente a Jeremy Bowers, registró una
entrada en Get Fit Now Gym el 9 de enero de 2018 a las 15:30 y una
salida a las 17:00.

Este resultado coincide con la declaración de Annabel Miller, quien
afirmó haber reconocido al asesino en su gimnasio mientras entrenaba
el 9 de enero.

Por tanto, la evidencia obtenida de ambos testigos apunta hacia Jeremy
Bowers.

# 🔎 Paso 7 — Verificar la información de Jeremy

Ahora vamos a obtener toda la información de Jeremy desde person y su licencia de conducir.

In [18]:
query = """
SELECT
    p.id AS person_id,
    p.name,
    p.license_id,
    p.address_number,
    p.address_street_name,
    p.ssn,
    dl.age,
    dl.height,
    dl.eye_color,
    dl.hair_color,
    dl.gender,
    dl.plate_number
FROM person AS p
JOIN drivers_license AS dl
    ON p.license_id = dl.id
WHERE p.id = 67318;
"""

jeremy = pd.read_sql_query(query, conn)

print(jeremy.to_string(index=False))

 person_id          name  license_id  address_number   address_street_name       ssn  age  height eye_color hair_color gender plate_number
     67318 Jeremy Bowers      423327             530 Washington Pl, Apt 3A 871539279   30      70     brown      brown   male       0H42W2


### La información obtenida es:

| Campo         | Resultado                   |
| ------------- | --------------------------- |
| `person_id`   | `67318`                     |
| `name`        | **Jeremy Bowers**           |
| `license_id`  | `423327`                    |
| Dirección     | `530 Washington Pl, Apt 3A` |
| Edad          | 30                          |
| Altura        | 70                          |
| Color de ojos | brown                       |
| Color de pelo | brown                       |
| Género        | male                        |
| Matrícula     | **`0H42W2`**                |


⚠️ Pero hay una última cosa importante

De momento solo encontramos el sospechoso.

El objetivo es resolver el Murder Mystery utilizando SQL y documentar las queries utilizadas.

Y tenemos una pista que todavía podemos utilizar para hacer una comprobación adicional:

Annabel dijo que reconoció al asesino.

Además, nuestra investigación ha encontrado a Jeremy mediante pistas independientes de Morty.

Lo ideal ahora es buscar si existe alguna entrevista de Jeremy Bowers. Esto nos permitirá comprobar si la base de datos contiene información adicional sobre él.

In [19]:
query = """
SELECT
    person_id,
    transcript
FROM interview
WHERE person_id = 67318;
"""

jeremy_interview = pd.read_sql_query(query, conn)

print(jeremy_interview.to_string(index=False))

 person_id                                                                                                                                                                                                                                         transcript
     67318 I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.\n


## Paso 8 — Identificar a la mujer que contrató a Jeremy

En su entrevista, Jeremy Bowers declara que fue contratado por una
mujer con las siguientes características:

- Altura aproximada entre 65 y 67 pulgadas.
- Pelo rojo.
- Conduce un Tesla Model S.

Utilizamos estas características para buscar posibles candidatas en
la tabla `drivers_license`.

La condición de altura se expresa mediante `BETWEEN 65 AND 67`,
incluyendo ambos extremos.

In [20]:
query = """
SELECT
    id,
    age,
    height,
    hair_color,
    gender,
    car_make,
    car_model,
    plate_number
FROM drivers_license
WHERE gender = 'female'
  AND height BETWEEN 65 AND 67
  AND hair_color = 'red'
  AND car_make = 'Tesla'
  AND car_model = 'Model S';
"""

potential_hires = pd.read_sql_query(query, conn)

print(potential_hires.to_string(index=False))

    id  age  height hair_color gender car_make car_model plate_number
202298   68      66        red female    Tesla   Model S       500123
291182   65      66        red female    Tesla   Model S       08CM64
918773   48      65        red female    Tesla   Model S       917UU3


# 🔎 Paso 9 — Obtener los nombres de las candidatas

Vamos a cruzar drivers_license con person.

In [21]:
query = """
SELECT
    p.id AS person_id,
    p.name,
    p.license_id,
    dl.age,
    dl.height,
    dl.hair_color,
    dl.gender,
    dl.car_make,
    dl.car_model,
    dl.plate_number
FROM person AS p
JOIN drivers_license AS dl
    ON p.license_id = dl.id
WHERE dl.id IN (202298, 291182, 918773);
"""

candidates = pd.read_sql_query(query, conn)

print(candidates.to_string(index=False))

 person_id             name  license_id  age  height hair_color gender car_make car_model plate_number
     99716 Miranda Priestly      202298   68      66        red female    Tesla   Model S       500123
     90700    Regina George      291182   65      66        red female    Tesla   Model S       08CM64
     78881         Red Korb      918773   48      65        red female    Tesla   Model S       917UU3


## Identificar a las candidatas

La consulta anterior identificó tres licencias de conducir que cumplen
las características proporcionadas por Jeremy Bowers.

Para conocer la identidad de estas personas, relacionamos la tabla
`drivers_license` con `person` mediante `person.license_id =
drivers_license.id`.

De esta forma obtenemos los nombres y los identificadores de las tres
candidatas.

# 🔎 Paso 10 — Utilizar la pista del concierto

Ahora tenemos la pista que nos permitirá diferenciarlas.

Jeremy dijo:

"She attended the SQL Symphony Concert 3 times in December 2017."

Tenemos que averiguar cuál de las tres candidatas asistió exactamente 3 veces al SQL Symphony Concert durante diciembre de 2017.

Aquí necesitamos inspeccionar la estructura de la base de datos para identificar dónde se registra la asistencia a eventos.

Como estamos trabajando de forma investigativa, antes de lanzar una query suponiendo el nombre de una tabla, vamos a comprobar las tablas disponibles.

In [22]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn)

print(tables.to_string(index=False))

                  name
    crime_scene_report
       drivers_license
facebook_event_checkin
  get_fit_now_check_in
    get_fit_now_member
                income
             interview
                person
              solution


### Investigar la asistencia al concierto

Las características físicas y del vehículo identifican tres candidatas:

- Miranda Priestly
- Regina George
- Red Korb

Sin embargo, Jeremy Bowers proporciona una pista adicional: la mujer
que lo contrató asistió tres veces al SQL Symphony Concert durante
diciembre de 2017.

Para localizar esta información, primero inspeccionamos las tablas
disponibles en la base de datos y buscamos aquella que registre
eventos o asistencias.

Inspeccionar facebook_event_checkin

SQLite nos permite consultar la estructura con PRAGMA:

In [23]:
query = """
PRAGMA table_info(facebook_event_checkin);
"""

table_info = pd.read_sql_query(query, conn)

print(table_info.to_string(index=False))

 cid       name    type  notnull dflt_value  pk
   0  person_id INTEGER        0       None   0
   1   event_id INTEGER        0       None   0
   2 event_name    TEXT        0       None   0
   3       date INTEGER        0       None   0


### Inspeccionar los registros de eventos

La base de datos contiene una tabla denominada
`facebook_event_checkin`, que presumiblemente almacena información
sobre la asistencia de personas a eventos.

Antes de realizar la consulta sobre el SQL Symphony Concert,
inspeccionamos la estructura de la tabla para identificar sus
columnas y construir la consulta utilizando los nombres reales de
la base de datos.

### Buscar las asistencias

Aquí hay una pequeña decisión metodológica importante: en lugar de buscar primero persona por persona, podemos consultar las tres candidatas simultáneamente y contar sus asistencias.

La fecha de diciembre de 2017 está entre:

20171201

y:

20171231

In [24]:
query = """
SELECT
    person_id,
    event_name,
    COUNT(*) AS veces_asistio
FROM facebook_event_checkin
WHERE person_id IN (99716, 90700, 78881)
  AND event_name = 'SQL Symphony Concert'
  AND date BETWEEN 20171201 AND 20171231
GROUP BY person_id, event_name
ORDER BY veces_asistio DESC;
"""

event_attendance = pd.read_sql_query(query, conn)

for _, row in event_attendance.iterrows():
    print(
        f"Person ID: {row['person_id']} | "
        f"Evento: {row['event_name']} | "
        f"Asistencias: {row['veces_asistio']}"
    )

Person ID: 99716 | Evento: SQL Symphony Concert | Asistencias: 3


Identificación de la persona que contrató a Jeremy

La última pista de Jeremy era:

La mujer que lo contrató asistió 3 veces al SQL Symphony Concert durante diciembre de 2017.

Nuestra consulta ha demostrado:

Person ID: 99716
Evento: SQL Symphony Concert
Asistencias: 3

Y anteriormente habíamos relacionado ese person_id con:

### `Miranda Priestly`

| Evidencia                 | Resultado                        |
| ------------------------- | -------------------------------- |
| Lugar del crimen          | SQL City                         |
| Fecha                     | 15/01/2018                       |
| Testigo 1                 | Morty Schapiro                   |
| Testigo 2                 | Annabel Miller                   |
| Bolsa del sospechoso      | Get Fit Now Gym                  |
| Membership                | `48Z55`                          |
| Estado                    | Gold                             |
| Sospechoso                | **Jeremy Bowers**                |
| Matrícula                 | `0H42W2`                         |
| Pista de matrícula        | Contiene `H42W` ✅                |
| Gimnasio                  | Jeremy estuvo allí el 09/01/2018 |
| Horario                   | 15:30–17:00                      |
| Contratante               | Mujer                            |
| Altura                    | 65–67"                           |
| Pelo                      | Rojo                             |
| Vehículo                  | Tesla Model S                    |
| Candidatas iniciales      | 3                                |
| Concierto                 | SQL Symphony Concert             |
| Asistencias requeridas    | 3                                |
| Persona con 3 asistencias | **Miranda Priestly**             |

🧩 Resultado final

Jeremy Bowers fue el asesino y Miranda Priestly fue la mujer que lo contrató.

## Conclusión — Murder Mystery

La investigación comenzó con el informe policial del asesinato ocurrido
en SQL City el 15 de enero de 2018.

El informe indicaba que había dos testigos. A partir de sus declaraciones
se obtuvieron las primeras pistas:

- Morty Schapiro vio a un hombre salir del lugar con una bolsa de
  "Get Fit Now Gym".
- El número de socio comenzaba por `48Z` y el miembro era Gold.
- El vehículo tenía una matrícula que contenía `H42W`.
- Annabel Miller declaró que reconoció al asesino de su gimnasio cuando
  estaba entrenando el 9 de enero.

La búsqueda de miembros Gold cuyo identificador comenzara por `48Z`
produjo dos candidatos: Joe Germuska y Jeremy Bowers.

Al cruzar estos candidatos con `drivers_license`, solamente Jeremy
Bowers tenía una matrícula que contenía `H42W`: `0H42W2`.

Posteriormente comprobamos que Jeremy Bowers, cuyo `membership_id` es
`48Z55`, estuvo en Get Fit Now Gym el 9 de enero de 2018 entre las
15:30 y las 17:00, coincidiendo con la pista proporcionada por Annabel.

Finalmente, la entrevista de Jeremy proporcionó una nueva pista:
afirmó que había sido contratado por una mujer con pelo rojo, de entre
65 y 67 pulgadas de altura, que conducía un Tesla Model S y que había
asistido tres veces al SQL Symphony Concert durante diciembre de 2017.

Estas características permitieron encontrar tres candidatas. Después
de comprobar sus asistencias al evento, Miranda Priestly
(`person_id = 99716`) fue la única candidata que asistió exactamente
tres veces al SQL Symphony Concert durante diciembre de 2017.

### Resultado

**Asesino: Jeremy Bowers**

**Persona que lo contrató: Miranda Priestly**